# Khaledoon — VoiceTut / OmniVoice Worker
شغّل الخلايا بالترتيب مرة واحدة بعد فتح الدفتر. العامل يسحب **مقطعًا واحدًا فقط** في كل مرة من طابور Supabase، ويعيد المحاولة عند الفشل. الملفات الناتجة تذهب إلى مساحة المراجعة الخاصة، ولا تُنشر تلقائيًا.

**مهم:** إذا انتهت جلسة Colab تمامًا فلن يستطيع Supabase إيقاظها. عند فتح الدفتر وتشغيل خلية العامل مجددًا، سيستأنف الطلبات المحفوظة تلقائيًا.

In [ ]:
%pip -q install git+https://github.com/k2-fsa/OmniVoice.git soundfile requests numpy
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, getpass, time, json, base64, hashlib, subprocess, threading, uuid
import numpy as np
import soundfile as sf
import requests

MODEL_ID = 'masterofaudio2077/voicetut-egyptian-tashkeel'
WORKER_URL = 'https://ypwprddkscnfblhftkeb.supabase.co/functions/v1/story-audio-worker'
WORKER_TOKEN = getpass.getpass('Khaledoon worker token: ').strip()
WORKER_ID = 'colab-' + str(uuid.uuid4())[:8]

# مسار مرجع كارما الموجود في Google Drive وفق إعداد المشروع.
KARMA_REF_AUDIO_PATH = '/content/drive/MyDrive/Karma_Voice/assets_audio_science_magdi-yacoub_ar_000.mp3'
# اتركه فارغًا لرفع مرجع الراوي من جهازك، أو ضع مساره في Drive.
NARRATOR_REF_AUDIO_PATH = ''

def ensure_reference(path, label):
    if path and os.path.isfile(path):
        return path
    print('ارفع ملف مرجع ' + label + ' الصوتي الآن.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('لم يتم رفع ملف مرجع ' + label + '.')
    return '/content/' + next(iter(uploaded.keys()))

KARMA_REF_AUDIO_PATH = ensure_reference(KARMA_REF_AUDIO_PATH, 'كارما')
NARRATOR_REF_AUDIO_PATH = ensure_reference(NARRATOR_REF_AUDIO_PATH, 'الراوي')

KARMA_REF_TEXT = input('اكتب النص المنطوق حرفيًا في مرجع كارما: ').strip()
NARRATOR_REF_TEXT = input('اكتب النص المنطوق حرفيًا في مرجع الراوي: ').strip()
if not WORKER_TOKEN or not KARMA_REF_TEXT or not NARRATOR_REF_TEXT:
    raise ValueError('يلزم إدخال مفتاح العامل ونصّي المرجعين.')
print('تم تجهيز إعدادات العامل. المفتاح محفوظ في ذاكرة الجلسة فقط.')

In [ ]:
import torch
from omnivoice import OmniVoice
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print('Loading', MODEL_ID, 'on', device)
voice_model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)
print('VoiceTut model loaded.')

In [ ]:
def worker_post(payload, timeout=180):
    response = requests.post(WORKER_URL, headers={
        'Content-Type': 'application/json',
        'x-khaledoon-worker-token': WORKER_TOKEN,
    }, json=payload, timeout=timeout)
    try:
        data = response.json()
    except Exception:
        data = {'error': response.text[:1000]}
    if not response.ok:
        raise RuntimeError(f'Worker HTTP {response.status_code}: {data}')
    return data

def as_numpy_audio(result):
    sample_rate = 24000
    audio = result
    if isinstance(result, dict):
        audio = result.get('audio', result.get('waveform', result.get('samples')))
        sample_rate = int(result.get('sample_rate', result.get('sr', sample_rate)))
    elif isinstance(result, (tuple, list)) and len(result) >= 2:
        audio, sample_rate = result[0], int(result[1])
    if hasattr(audio, 'detach'):
        audio = audio.detach().cpu().numpy()
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim == 2 and audio.shape[0] <= 2 and audio.shape[1] > audio.shape[0]:
        audio = audio.T
    if audio.ndim not in (1, 2) or audio.size == 0:
        raise ValueError('Unexpected audio tensor returned by VoiceTut.')
    return audio, sample_rate

def make_mp3(job):
    payload = job.get('payload') or {}
    speaker = job['speaker']
    language = payload.get('language_code') or ('arz' if job['language'] == 'ar' else 'English')
    if speaker == 'karma':
        ref_audio, ref_text = KARMA_REF_AUDIO_PATH, KARMA_REF_TEXT
    else:
        ref_audio, ref_text = NARRATOR_REF_AUDIO_PATH, NARRATOR_REF_TEXT
    text_to_speak = str(payload.get('text') or '').strip()
    if not text_to_speak:
        raise ValueError('Queue job has empty speech text.')
    generated = voice_model.generate(
        text_to_speak,
        language=language,
        ref_audio=ref_audio,
        ref_text=ref_text,
        num_step=36,
        guidance_scale=3.0,
        speed=0.97,
    )
    audio, sample_rate = as_numpy_audio(generated)
    stem = f"{job['section']}-{job['slug']}-{job['language']}-{int(job['block_index']):04d}"
    wav_path, mp3_path = f'/content/{stem}.wav', f'/content/{stem}.mp3'
    sf.write(wav_path, audio, sample_rate)
    subprocess.run(['ffmpeg','-y','-i',wav_path,'-codec:a','libmp3lame','-b:a','128k',mp3_path],check=True,capture_output=True)
    with open(mp3_path,'rb') as f:
        mp3_bytes = f.read()
    if not mp3_bytes or len(mp3_bytes) > 13_500_000:
        raise ValueError('Generated MP3 is empty or exceeds the worker upload limit.')
    return mp3_path, mp3_bytes

print('Worker helpers ready.')

In [ ]:
stop_worker = False
def run_worker():
    global stop_worker
    print('Worker online:', WORKER_ID)
    while not stop_worker:
        job = None
        try:
            claimed = worker_post({'action':'claim','worker_id':WORKER_ID,'lease_seconds':900})
            job = claimed.get('job')
            if not job:
                time.sleep(6)
                continue
            print('Claimed', job['section'], job['slug'], job['language'], 'block', job['block_index'], job['speaker'])
            lease_stop = threading.Event()
            def heartbeat_loop():
                while not lease_stop.wait(90):
                    try:
                        worker_post({'action':'heartbeat','job_id':job['id'],'lock_token':job['lock_token'],'worker_id':WORKER_ID}, timeout=30)
                    except Exception as heartbeat_error:
                        print('Heartbeat warning:', heartbeat_error)
            heartbeat_thread = threading.Thread(target=heartbeat_loop, daemon=True)
            heartbeat_thread.start()
            try:
                mp3_path, mp3_bytes = make_mp3(job)
                result = worker_post({
                    'action':'complete','job_id':job['id'],'lock_token':job['lock_token'],
                    'extension':'mp3','content_type':'audio/mpeg',
                    'audio_base64':base64.b64encode(mp3_bytes).decode('ascii'),
                    'sha256':hashlib.sha256(mp3_bytes).hexdigest(),
                }, timeout=240)
                print('Completed:', result.get('job_id'), result.get('status'))
            finally:
                lease_stop.set()
                heartbeat_thread.join(timeout=2)
        except Exception as error:
            print('Job/worker error:', repr(error))
            if job and job.get('id') and job.get('lock_token'):
                try:
                    worker_post({'action':'fail','job_id':job['id'],'lock_token':job['lock_token'],'error':str(error)[:1800]}, timeout=30)
                except Exception as fail_error:
                    print('Could not acknowledge failure (lease may have expired):', fail_error)
            time.sleep(5)
    print('Worker stopped.')

run_worker()

## إيقاف العامل
لتوقيف حلقة العمل يدويًا، أوقف تنفيذ الخلية من Colab. الطلب الحالي سيُعاد إلى الطابور إذا انتهت مهلة الحجز، وسيُلتقط عند تشغيل العامل مرة أخرى.